# Projet 3 · Compétition Kaggle « Titanic » · ⭐⭐⭐

**Bloc 3 · Niveau ⭐⭐⭐ Avancé** · Le grand projet du parcours, en binôme sur deux séances.

Le 15 avril 1912, le Titanic coule. Kaggle te donne la fiche de 891 passagers **avec** la réponse (survécu ou non) et te demande de **prédire** la survie de 418 autres. Ton score s'affiche sur un classement mondial. Ce notebook est le fil complet : préparer les données, comparer **3 modèles**, tenir un **journal des essais**, générer la **soumission**.

Comment travailler :
- Google Colab, `Maj + Entrée`. En binôme : un tape, l'autre réfléchit à voix haute, on échange toutes les 20 minutes.
- Les leçons 7 et 8 expliquent chaque étape ; ici on va droit au projet.
- Sans les fichiers Kaggle, une copie publique de `train.csv` est chargée automatiquement et un faux `test.csv` est fabriqué : tout fonctionne, sauf la soumission réelle.


## Préparation : les fichiers Kaggle

1. Compte Kaggle (gratuit) → https://www.kaggle.com/competitions/titanic → **Join Competition**.
2. Onglet **Data** → **Download All** → dépose `train.csv` et `test.csv` dans Colab (icône dossier à gauche). Ou bien utilise l'API `kaggle` (voir le README du dossier `projets/`).
3. Variante plus ludique avec les mêmes étapes : https://www.kaggle.com/competitions/spaceship-titanic (colonnes différentes, à adapter).

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL_SECOURS = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"   # = train.csv

if os.path.exists("train.csv"):
    train = pd.read_csv("train.csv")
    print("train.csv de Kaggle chargé")
else:
    try:
        train = pd.read_csv(URL_SECOURS)
        print("Copie publique de train.csv chargée")
    except Exception as erreur:
        print("Impossible de charger les données : pas de réseau ?", erreur)
        raise

if os.path.exists("test.csv"):
    test = pd.read_csv("test.csv")
    SOUMISSION_REELLE = True
    print("test.csv de Kaggle chargé :", len(test), "passagers à prédire")
else:
    # Pas de test.csv : on met 20 % de train de côté pour simuler la compétition (et connaître le vrai score)
    SOUMISSION_REELLE = False
    test = train.sample(frac=0.2, random_state=0)
    train = train.drop(test.index)
    reponses_cachees = test["Survived"].values
    test = test.drop(columns="Survived")
    print("test.csv absent : faux test fabriqué avec", len(test), "passagers (score local seulement)")

print(train.shape, test.shape)
train.head(3)

## 1. Explorer vite, poser 3 hypothèses (20 min)

Tu l'as fait en détail à la séance 7. Ici, 3 chiffres pour te remettre en tête ce qui compte : le sexe, la classe, l'âge.

In [ ]:
print("Survie globale :", round(train["Survived"].mean() * 100, 1), "%\n")
print(train.groupby("Sex")["Survived"].mean().round(2), "\n")
print(train.groupby("Pclass")["Survived"].mean().round(2), "\n")
print("Cases vides :", train.isna().sum()[train.isna().sum() > 0].to_dict())

In [ ]:
HYPOTHESES = [
    "Les femmes ont survécu bien plus souvent que les hommes.",
    "La 1re classe a été mieux sauvée que la 3e.",
    "Les enfants ont eu plus de chances de survivre.",
]
for h in HYPOTHESES:
    print("-", h)

## 2. Préparer les données : une fonction unique (30 min)

Tout ce qu'on fait à `train`, on doit le faire **exactement pareil** à `test`. D'où une seule fonction `preparer()`. Elle :
- extrait le **titre** du nom avec une regex (`Mr`, `Mrs`, `Miss`, `Master`, autres) ;
- remplit l'âge manquant par la **médiane du titre** (un « Master » est un garçon, pas un homme de 40 ans) ;
- crée `Famille` (= SibSp + Parch + 1) et `Seul` ;
- remplit `Embarked` et `Fare`, encode `Sex` et `Embarked` en nombres.

In [ ]:
def preparer(df, medianes_age=None):
    """Transforme le dataframe brut en tableau de nombres pour le modèle. Renvoie (X, medianes_age)."""
    d = df.copy()
    d["Titre"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
    d["Titre"] = d["Titre"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    d.loc[~d["Titre"].isin(["Mr", "Mrs", "Miss", "Master"]), "Titre"] = "Autre"

    if medianes_age is None:                       # calculées sur train, réutilisées pour test
        medianes_age = d.groupby("Titre")["Age"].median()
    d["Age"] = d["Age"].fillna(d["Titre"].map(medianes_age))
    d["Age"] = d["Age"].fillna(medianes_age.median())

    d["Famille"] = d["SibSp"] + d["Parch"] + 1
    d["Seul"] = (d["Famille"] == 1).astype(int)
    d["Embarked"] = d["Embarked"].fillna("S")
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Sexe"] = (d["Sex"] == "female").astype(int)
    d["Enfant"] = (d["Age"] < 12).astype(int)

    colonnes = ["Pclass", "Sexe", "Age", "Fare", "Famille", "Seul", "Enfant"]
    X = d[colonnes].copy()
    for port in ["C", "Q", "S"]:                   # one-hot : une colonne par port
        X[f"Port_{port}"] = (d["Embarked"] == port).astype(int)
    for titre in ["Mr", "Mrs", "Miss", "Master", "Autre"]:
        X[f"Titre_{titre}"] = (d["Titre"] == titre).astype(int)
    return X, medianes_age

X_train, medianes = preparer(train)
y_train = train["Survived"]
X_test, _ = preparer(test, medianes)
print(X_train.shape, X_test.shape, "· cases vides restantes :", int(X_train.isna().sum().sum() + X_test.isna().sum().sum()))
X_train.head(3)

**À toi** : ajoute une variable dans `preparer()`. Idées : `Cabine_connue` (Cabin non vide), `Fare_par_personne` (Fare / Famille), une tranche de `Fare` (`pd.qcut`), la lettre du pont (`Cabin.str[0]`). Relance la cellule ci-dessus puis les modèles : le score bouge-t-il ?

<details><summary>Solution (Cabine_connue)</summary>

```python
# dans preparer(), avant `colonnes = [...]` :
d["Cabine_connue"] = d["Cabin"].notna().astype(int)
# puis ajoute "Cabine_connue" à la liste `colonnes`
```
</details>

In [ ]:
# À toi : modifie preparer() ci-dessus, puis note ici les variables que tu as ajoutées
VARIABLES_AJOUTEES = []     # ex. ["Cabine_connue"]
print("Variables ajoutées :", VARIABLES_AJOUTEES or "aucune pour l'instant")

## 3. Trois modèles et la validation croisée (30 min)

On ne peut pas mesurer sur `test` (pas de réponses). On utilise la **validation croisée** : on découpe `train` en 5 morceaux, on apprend sur 4 et on mesure sur le 5e, 5 fois de suite. Le score obtenu est une bonne estimation du score Kaggle. On compare aussi le score sur les données d'apprentissage : s'il est bien plus haut, le modèle **sur-apprend** (il a appris par cœur).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MODELES = {
    "régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "forêt aléatoire":       RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "gradient boosting":     GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}

def evaluer(nom, modele, X, y):
    cv = cross_val_score(modele, X, y, cv=5)
    modele.fit(X, y)
    apprentissage = modele.score(X, y)
    print(f"{nom:24s} validation croisée : {cv.mean()*100:.1f} % (± {cv.std()*100:.1f})   apprentissage : {apprentissage*100:.1f} %")
    return cv.mean(), apprentissage

scores = {nom: evaluer(nom, m, X_train, y_train) for nom, m in MODELES.items()}

In [ ]:
noms = list(scores)
plt.figure(figsize=(7, 3.5))
plt.barh(noms, [scores[n][0] * 100 for n in noms], label="validation croisée")
plt.barh(noms, [scores[n][1] * 100 for n in noms], height=0.4, alpha=0.5, label="apprentissage")
plt.xlim(70, 100); plt.xlabel("% de bonnes prédictions"); plt.legend(); plt.tight_layout(); plt.show()

## 4. Le journal des essais (tout au long des 2 séances)

Un data scientist note **chaque essai** : sinon on refait deux fois la même chose et on ne sait plus ce qui a marché. Appelle `noter_essai(...)` après chaque tentative ; le score Kaggle se remplit après soumission.

In [ ]:
JOURNAL_ESSAIS = []

def noter_essai(nom, variables, modele, score_cv, score_kaggle=None, commentaire=""):
    JOURNAL_ESSAIS.append({"essai": len(JOURNAL_ESSAIS) + 1, "nom": nom, "variables": variables, "modèle": modele,
                           "score CV": round(score_cv * 100, 1), "score Kaggle": score_kaggle, "commentaire": commentaire})
    return pd.DataFrame(JOURNAL_ESSAIS)

base = "Pclass, Sexe, Age, Fare, Famille, Seul, Enfant, Port, Titre"
for nom in MODELES:
    noter_essai(f"{nom} de base", base, nom, scores[nom][0])
pd.DataFrame(JOURNAL_ESSAIS)

**À toi** : au moins **3 essais supplémentaires** dans le journal. Idées : (1) une nouvelle variable (section 2) ; (2) régler la forêt (`max_depth` 4, 6, 10, `None`) et observer l'écart apprentissage / validation ; (3) enlever une variable (`Fare` ?) pour voir si elle sert.

<details><summary>Indice</summary>

```python
for profondeur in [3, 6, 10, None]:
    m = RandomForestClassifier(n_estimators=300, max_depth=profondeur, random_state=42)
    cv, app = evaluer(f"forêt profondeur {profondeur}", m, X_train, y_train)
    noter_essai(f"forêt profondeur {profondeur}", base, "forêt aléatoire", cv,
                commentaire="sur-apprentissage" if app - cv > 0.1 else "")
pd.DataFrame(JOURNAL_ESSAIS)
```
</details>

In [ ]:
# À toi : tes essais
for profondeur in [3, 10]:
    m = RandomForestClassifier(n_estimators=300, max_depth=profondeur, random_state=42)
    cv, app = evaluer(f"forêt profondeur {profondeur}", m, X_train, y_train)
    noter_essai(f"forêt profondeur {profondeur}", base, "forêt aléatoire", cv,
                commentaire="sur-apprentissage" if app - cv > 0.1 else "")
pd.DataFrame(JOURNAL_ESSAIS)

## 5. La soumission (15 min)

On choisit le modèle au meilleur score de validation croisée, on prédit `test`, on écrit `submission.csv` (colonnes `PassengerId`, `Survived`, 418 lignes), et on l'envoie sur la page de la compétition (**Submit Prediction**). Note le score Kaggle dans le journal.

In [ ]:
meilleur_nom = max(scores, key=lambda n: scores[n][0])
meilleur = MODELES[meilleur_nom].fit(X_train, y_train)
predictions = meilleur.predict(X_test)

soumission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions.astype(int)})
soumission.to_csv("submission.csv", index=False)
print("Modèle choisi :", meilleur_nom)
print("submission.csv écrit :", len(soumission), "lignes · part de survivants prédite :", round(predictions.mean() * 100, 1), "%")

if SOUMISSION_REELLE:
    print("→ Va sur https://www.kaggle.com/competitions/titanic/submissions et dépose submission.csv")
else:
    score_local = (predictions == reponses_cachees).mean()
    print(f"(pas de test.csv) Score sur le faux test : {score_local*100:.1f} %  ← c'est ce que Kaggle t'afficherait")
soumission.head()

In [ ]:
SCORE_KAGGLE = None       # ← ex. 0.77990, recopié depuis Kaggle après soumission
if SCORE_KAGGLE is not None:
    JOURNAL_ESSAIS[-1]["score Kaggle"] = SCORE_KAGGLE
pd.DataFrame(JOURNAL_ESSAIS)

## 6. Présenter sa démarche (5 min par binôme)

Plan : l'hypothèse la plus surprenante · la variable ajoutée et ce qu'elle a changé · le tableau des essais · le score final et **ce que vous feriez avec une séance de plus**. Un score honnête de 0,77-0,80 est très bien : au-delà de 0,83, méfiez-vous des notebooks qui trichent avec les réponses publiques.

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

verifier("Aucune case vide après preparer()", int(X_train.isna().sum().sum()) == 0)
verifier("train et test ont les mêmes colonnes", list(X_train.columns) == list(X_test.columns))
verifier("3 modèles évalués en validation croisée", len(scores) >= 3)
verifier("Au moins 6 essais dans le journal", len(JOURNAL_ESSAIS) >= 6)
verifier("submission.csv a le bon format", list(soumission.columns) == ["PassengerId", "Survived"] and len(soumission) == len(test))
verifier("Une variable ajoutée par le binôme", len(VARIABLES_AJOUTEES) > 0)
verifier("Score Kaggle noté", SCORE_KAGGLE is not None)

## Pour aller plus loin
- **Spaceship Titanic** : même démarche, dataset plus récent et plus « propre » à sur-apprendre : https://www.kaggle.com/competitions/spaceship-titanic
- `GridSearchCV` pour chercher automatiquement les meilleurs réglages de la forêt.
- Lis le notebook de référence de Manav Sehgal (lien dans le README) : il fait le même travail avec 10 variables de plus. Laquelle te semble la plus utile ?